# Clinical Document Classification — Sequence Models

This notebook covers the **deep-learning sequence model** branch (thesis 4.3.2 "Sequence Models",
5.2.4(2)). Same 5 classes and same text-cleaning pipeline as
`01_traditional_ml_classification.ipynb`, but features are learned word embeddings fed through
recurrent layers instead of TF-IDF + a classical classifier.

Architecture (thesis 5.2.4): `Embedding -> Dropout -> [recurrent layer (128 units)] -> Dropout(0.5)
-> [recurrent layer (64 units)] -> Dense(5, softmax)`, compiled with Adam / categorical cross-entropy,
trained with 5-fold cross-validation, 6 epochs per fold, early stopping on validation loss
(patience 3). The same builder is used for SimpleRNN, LSTM, and GRU so the three are compared on
identical footing (thesis Table 5.12).

In [ ]:
import sys
sys.path.append('src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, SimpleRNN, LSTM, GRU
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics

from preprocessing import load_clinical_documents, load_spacy_model, preprocess_document

RANDOM_STATE = 133
DATA_DIR = "../data/clinical_documents"

# Thesis 5.2.4(2) hyperparameters
VOCAB_SIZE = 10_000
NUM_FEATURES = 5_000
INPUT_LENGTH = 150
EMBEDDING_DIM = 50
N_FOLDS = 5
EPOCHS_PER_FOLD = 6
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 3

## 1. Load & preprocess data

Same four-step cleaning pipeline as the traditional-ML notebook (`src/preprocessing.py`), so results
across notebooks stay comparable. If you already ran notebook 1 in this session, this repeats the
~1-2 minute cleaning pass rather than reusing its output, to keep this notebook runnable standalone.

In [ ]:
df = load_clinical_documents(DATA_DIR)
nlp = load_spacy_model("en_core_web_lg")

df["clean_document"] = [preprocess_document(doc, nlp) for doc in df["document"]]
df[["label", "clean_document"]].head()

## 2. Tokenization & padding

Unlike TF-IDF, sequence models need integer token sequences of a fixed length: a Keras `Tokenizer`
builds the vocabulary (capped at `VOCAB_SIZE`), `pad_sequences` truncates/pads every document to
`INPUT_LENGTH` tokens, and labels are one-hot encoded for the softmax output layer.

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE)
tokenizer.fit_on_texts(df["clean_document"])

sequences = tokenizer.texts_to_sequences(df["clean_document"])
X = pad_sequences(sequences, maxlen=INPUT_LENGTH)

label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(df["label"])
Y = to_categorical(y_int, num_classes=len(label_encoder.classes_))

print("X:", X.shape, " Y:", Y.shape)

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=RANDOM_STATE, shuffle=True)

## 3. Custom Keras metrics

Precision/recall/F1 aren't built into Keras by default; these mirror what's used for evaluation elsewhere in the project.

In [ ]:
precision_metric = tf.keras.metrics.Precision()
recall_metric = tf.keras.metrics.Recall()

def precision(y_true, y_pred):
    return precision_metric(y_true, tf.round(y_pred))

def recall(y_true, y_pred):
    return recall_metric(y_true, tf.round(y_pred))

def f1_score_metric(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * ((p * r) / (p + r + K.epsilon()))

## 4. Model builder

One function builds any of the three architectures with the same shape, so RNN/LSTM/GRU are
compared like-for-like.

In [ ]:
LAYER_TYPES = {"RNN": SimpleRNN, "LSTM": LSTM, "GRU": GRU}

def build_sequence_model(layer_name):
    layer_cls = LAYER_TYPES[layer_name]
    model = Sequential([
        Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=INPUT_LENGTH),
        Dropout(0.2),
        layer_cls(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),
        Dropout(0.5),
        layer_cls(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(Y.shape[1], activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy",
                  metrics=["accuracy", precision, recall, f1_score_metric])
    return model

build_sequence_model("GRU").summary()

## 5. Train & compare RNN / LSTM / GRU (5-fold CV)

Matches thesis Table 5.12. Each architecture is trained with 5-fold cross-validation on the
training split, 6 epochs per fold with early stopping (patience 3, monitoring validation loss);
fold metrics are averaged.

In [ ]:
def cross_validate(layer_name, X_data, Y_data):
    kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_data), start=1):
        model = build_sequence_model(layer_name)
        early_stop = EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

        model.fit(
            X_data[train_idx], Y_data[train_idx],
            validation_data=(X_data[val_idx], Y_data[val_idx]),
            epochs=EPOCHS_PER_FOLD, batch_size=BATCH_SIZE,
            callbacks=[early_stop], verbose=0,
        )

        y_pred = np.argmax(model.predict(X_data[val_idx], verbose=0), axis=1)
        y_true = np.argmax(Y_data[val_idx], axis=1)

        fold_metrics.append({
            "accuracy": metrics.accuracy_score(y_true, y_pred),
            "precision": metrics.precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall": metrics.recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1": metrics.f1_score(y_true, y_pred, average="macro", zero_division=0),
        })
        print(f"{layer_name} fold {fold}/{N_FOLDS}: acc={fold_metrics[-1]['accuracy']:.3f} f1={fold_metrics[-1]['f1']:.3f}")

    return pd.DataFrame(fold_metrics).mean().to_dict()

In [ ]:
sequence_results = {}
for layer_name in ["RNN", "LSTM", "GRU"]:
    print(f"\n=== Cross-validating {layer_name} ===")
    sequence_results[layer_name] = cross_validate(layer_name, x_train, y_train)

sequence_results_df = pd.DataFrame(sequence_results).T
display(sequence_results_df)

sequence_results_df[["accuracy", "f1"]].plot(kind="bar", figsize=(6, 4), title="Sequence model comparison (5-fold CV)")
plt.ylim(0, 1)
plt.show()

## 6. Best model (GRU) — full evaluation

GRU is the best-performing sequence architecture (thesis: catches longer-term dependencies than a
plain RNN and converges faster than an LSTM before overfitting sets in). Trained on the full
training split and evaluated on the held-out test set (thesis Table 5.13).

In [ ]:
gru_model = build_sequence_model("GRU")
early_stop = EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

history = gru_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS_PER_FOLD, batch_size=BATCH_SIZE,
    callbacks=[early_stop], verbose=2,
)

In [ ]:
y_pred = np.argmax(gru_model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
target_names = label_encoder.classes_

print(metrics.classification_report(y_true, y_pred, target_names=target_names))

cm = metrics.confusion_matrix(y_true, y_pred)
sns.heatmap(cm, center=True, cmap="YlGnBu", xticklabels=target_names, yticklabels=target_names)
plt.title("GRU — confusion matrix")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="validation accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()
plt.title("GRU training curve")
plt.show()

## 7. Save the best sequence model

In [ ]:
import os
import pickle

os.makedirs("models", exist_ok=True)

gru_model.save("models/gru_sequence_model.keras")
with open("models/sequence_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("models/sequence_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Saved GRU model, tokenizer, and label encoder to notebooks/models/.")